In [1]:
import pandas as pd 

In [2]:
trips = pd.read_csv(r"C:\works\learnings\Advance_EDA\Trips.csv")
users = pd.read_csv(r"C:\works\learnings\Advance_EDA\Users.csv")

In [3]:
trips

,id,client_id,driver_id,city_id,status,request_at
0,1,1,10,1,completed,2013-10-01
1,2,2,11,1,cancelled_by_driver,2013-10-01
2,3,3,12,6,completed,2013-10-01
3,4,4,13,6,cancelled_by_client,2013-10-01
4,5,1,10,1,completed,2013-10-02
5,6,2,11,6,completed,2013-10-02
6,7,3,12,6,completed,2013-10-02
7,8,2,12,12,completed,2013-10-03
8,9,3,10,12,completed,2013-10-03
9,10,4,13,12,cancelled_by_driver,2013-10-03


In [4]:
users

,users_id,banned,role
0,1,No,client
1,2,Yes,client
2,3,No,client
3,4,No,client
4,10,No,driver
5,11,No,driver
6,12,No,driver
7,13,No,driver


In [54]:
def solve(item):
    if item in ["cancelled_by_client", "cancelled_by_driver"] :
        return 1
    else :
        return None

In [55]:
df = trips.merge(users, left_on = "client_id", right_on="users_id", how="inner")
df = df.merge(users, left_on="driver_id", right_on= "users_id", how = "inner")
df = df[(df["banned_x"] == "No") & (df["banned_y"] == "No")]

In [ ]:

df["cancel_count"] = df["status"].apply(solve)
each_day_count = df.groupby("request_at")["status"].count()
df = df.groupby("request_at").agg({
    "cancel_count" : ["count"],
    "status" : ["count"]
 }).reset_index()

df.columns = [ "_".join(col) for col in df.columns]


,request_at_,cancel_count_count,status_count
0,2013-10-01,1,3
1,2013-10-02,0,2
2,2013-10-03,1,2


In [57]:
df["cancel_pct"] = df["cancel_count_count"] / df["status_count"]
df

,request_at_,cancel_count_count,status_count,cancel_pct
0,2013-10-01,1,3,0.333333
1,2013-10-02,0,2,0.000000
2,2013-10-03,1,2,0.500000


In [1]:
import pandas as pd 

players = pd.read_csv(r"C:\works\learnings\Advance_EDA\players.csv")
matches = pd.read_csv(r"C:\works\learnings\Advance_EDA\matches.csv")

players

,player_id,group_id
0,15,1
1,25,1
2,30,1
3,45,1
4,10,2
5,35,2
6,50,2
7,20,3
8,40,3


In [2]:
matches

,match_id,first_player,second_player,first_score,second_score
0,1,15,45,3,0
1,2,30,25,1,2
2,3,30,15,2,0
3,4,40,20,5,2
4,5,35,50,1,1


In [8]:
p1 = matches[['first_player', 'first_score']].rename(columns={'first_player': 'player_id', 'first_score': 'score'})
p2 = matches[['second_player', 'second_score']].rename(columns={'second_player': 'player_id', 'second_score': 'score'})

result = pd.concat([p1, p2], ignore_index=True)
result = result.groupby("player_id")["score"].sum().reset_index()
result

,player_id,score
0,15,3
1,20,2
2,25,2
3,30,3
4,35,1
5,40,5
6,45,0
7,50,1


In [17]:
df

,player_id,score,group_id
0,15,3,1
1,20,2,3
2,25,2,1
3,30,3,1
4,35,1,2
5,40,5,3
6,45,0,1
7,50,1,2


In [20]:
df = result.merge(players, on="player_id", how="left")

# total score per player
player_scores = df.groupby(["group_id", "player_id"])["score"].sum().reset_index()

# max score per group
max_scores = player_scores.groupby("group_id")["score"].max().reset_index()
max_scores.columns = ["group_id", "max_score"]

# join back to get the player_id, handle ties by picking lowest player_id
merged = player_scores.merge(max_scores, on="group_id")
winners = merged[merged["score"] == merged["max_score"]]
winners = winners.sort_values("player_id").groupby("group_id").first().reset_index()

In [22]:
player_scores

,group_id,player_id,score
0,1,15,3
1,1,25,2
2,1,30,3
3,1,45,0
4,2,35,1
5,2,50,1
6,3,20,2
7,3,40,5


In [23]:
merged

,group_id,player_id,score,max_score
0,1,15,3,3
1,1,25,2,3
2,1,30,3,3
3,1,45,0,3
4,2,35,1,1
5,2,50,1,1
6,3,20,2,5
7,3,40,5,5


In [21]:
winners

,group_id,player_id,score,max_score
0,1,15,3,3
1,2,35,1,1
2,3,40,5,5
